## 1. DETR 推理代码运行理解
* 两阶段目标检测：1 区域提议阶段，即确定很大很多的anchor（框框）；2 分类与回归阶段，非极大值抑制。

* DETR（DEtection TRansformer）是一种端到端目标检测模型，核心结构是 CNN + Transformer。

* 推理过程主要包括：

  1. 图像输入 → Backbone（如 ResNet）提取特征；
  2. 特征图 → Position Embedding 添加位置信息；
  3. Transformer Encoder-Decoder 处理；
  4. 输出 query 与每个位置匹配 → 生成类别 + 边界框坐标；
  5. 选择 top-k 得分结果作为最终预测。

## 2. 理解 Tensor 尺寸变化（关键）

以下是典型输入输出维度流程：

| 模块                           | Tensor 尺寸变化（假设 batch=1）                                          |
| ---------------------------- | ---------------------------------------------------------------- |
| 输入图像                         | `[3, H, W]`                                                      |
| Backbone网络ResNet101提取图像的高层特征                    | `[C, H', W']`（如 `[2048, 64, 64]`）                                |
| Flatten + Position Embedding | `[H'*W', C]` → `[N, d_model]`                                    |
| Transformer Encoder 输出       | `[N, d_model]`                                                   |
| Decoder 输入：query embedding   | `[hidden_dim, d_model]`                                         |
| Decoder 输出                   | `[hidden_dim, d_model]`                                         |
| FFN（预测分类 + 框）                | `logits: [hidden_dim, num_classes+1]`，`boxes: [hidden_dim, 4]` |

## 3. 编写 DETR 模型结构 + 损失计算笔记（简要）

### 模型结构

```text
Input → CNN (ResNet) → Flatten → 结合位置编码（Linear_pro得到hidden_dim） → Transformer Encoder （随机作为query喂入Decoder与EO做交叉注意力）→ Transformer Decoder → FFN → 分类+回归
```

### 损失函数（Hungarian Matching Loss）

* 使用匈牙利算法进行一一匹配,即选择最合适的anchor
* 计算总损失由以下几部分组成：

  * 分类损失（CrossEntropy）
  * 边界框 L1 损失
  * GIoU 损失

$$
loss = loss_{cls} + λ_1 * loss_{L1} + λ_2 * loss_{giou}
$$




## 评估指标

- **AP指标**：AP即平均精度，是目标检测中的一个常用指标。它是精确率-召回率（PR）曲线下的面积。在多类别目标检测任务中，通常会计算mAP（mean Average Precision），即所有类别的AP的平均值。

- **其他相关评估指标**：除了AP即mAP外，DETR还可能使用其他评估指标，如AP@IoU=0.50、AP@IoU=0.75等，分别表示在交并比（IoU）阈值为0.50和0.75时的平均精度，以及评估不同大小物体的目标检测精度APs,APm,APL。此外，还有AR（Average Recall）指标，衡量的是在不同检测框数量限制下的召回率，如AR@maxDets=1、AR@maxDets=10等，表示最多允许检测1个框和10个框时的平均召回率。

- **GFLOPS**:一次正向计算，模型消耗的计算次数

- **FPS**:每秒钟能够处理的图片个数

## 消融实验与可视化

### 一、消融实验设计
通过控制变量法验证各组件作用：
1. **Transformer架构**：
   - 调整编码器/解码器层数，发现性能随层数增加提升但边际效益递减；
   - 验证自注意力与交叉注意力的必要性，移除交叉注意力会导致目标对齐失效。
2. **集合预测机制**：
   - 证明匈牙利匹配对消除冗余预测的关键作用，缺失会导致重复标注；
   - 对比不同分类损失（如Focal Loss）和边界框损失（GIoU）的影响。
3. **骨干网络与特征**：
   - DC5是DETR中骨干网络的一种变体，它在CNN的最后阶段增加了膨胀（dilated）操作，也称为扩张C5。这种操作可以增大感受域，提高特征图的分辨率，从而提升预测效果。基于DC5的DETR模型有DETR-DC5、DETR-DC5-R50和DETR-DC5-R101等，其中DETR-DC5-R101在COCO数据集上取得了较好的性能表现。DC5可提升小目标AP约3-5%；
   - 验证多尺度特征对检测精度的增益。
4. **训练策略**：
   - 分析预训练、训练时长及解码器中间监督（辅助损失）对稳定性的影响。


### 二、可视化分析
1. **注意力可视化**：
   - 解码器交叉注意力：展示查询与图像区域的对齐（如“猫”类查询聚焦猫的区域）；
   - 编码器自注意力：揭示图像长程依赖（如关联人与手持物品）。
2. **预测与匹配过程**：
   - 跟踪预测框从初始化到收敛的演化；
   - 可视化匈牙利匹配如何将预测框与真实标签关联。
3. **特征与失败案例**：
   - t-SNE降维显示类别特征聚类；
   - 分析误检（背景误判）和漏检（小目标、遮挡）案例。


### 核心发现
Transformer的全局建模能力使DETR在大目标检测上优于传统方法，集合预测机制可省却NMS；DC5等模块有效提升小目标性能，但模型在密集场景仍有局限。